# Track A pseudo-pair 10-epoch duration probe

This unexecuted Colab launcher changes only training duration relative to the micro-v2 contract. It reuses an already observed split as development evidence, evaluates selected slices once at epoch 10, and cannot support a confirmatory or complete-volume claim. The scaled pilot remains blocked.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

import re
from pathlib import Path

REPO_URL = "https://github.com/GuillermoTafoya/MRIxFields.git"
EXPECTED_SPLIT_SHA256 = "17f00411ab04331fa0380526b2d8f0cd0173e4ff6f8978f72c61053fa7385dbe"
EXPECTED_EPOCHS = 10
EXPECTED_STEPS_PER_EPOCH = 16
EXPECTED_GLOBAL_STEPS = 160

EXPECTED_CODE_COMMIT = input("Exact duration-probe Git commit SHA: " ).strip()
MANIFEST_PATH = Path(input("External real-data manifest path: " ).strip()).expanduser().resolve()
PRIOR_SPLIT_PATH = Path(input("Prior Drive volume_splits.json path: " ).strip()).expanduser().resolve()
RUN_DIR = Path(input("New Drive run directory (must not exist): " ).strip()).expanduser().resolve()

if re.fullmatch(r"[0-9a-f]{40}", EXPECTED_CODE_COMMIT) is None:
    raise ValueError("EXPECTED_CODE_COMMIT must be an exact 40-character lowercase Git SHA.")
if not MANIFEST_PATH.is_file():
    raise FileNotFoundError("The external manifest path does not exist.")
if PRIOR_SPLIT_PATH.name != "volume_splits.json" or not PRIOR_SPLIT_PATH.is_file():
    raise FileNotFoundError("Provide the prior Drive volume_splits.json file.")
if RUN_DIR.exists():
    raise FileExistsError("Fresh initialization requires a new run directory.")
RUN_DIR.mkdir(parents=True)


In [ ]:
import importlib
import os
import subprocess
import sys

REPO_DIR = (Path.cwd() / "FieldBridge-duration-probe").resolve()
if REPO_DIR.exists():
    raise FileExistsError("Use a fresh Colab runtime or remove the old checkout manually.")
if RUN_DIR == REPO_DIR or REPO_DIR in RUN_DIR.parents:
    raise ValueError("Run outputs must remain outside the Git checkout.")

subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "checkout", "--detach", EXPECTED_CODE_COMMIT], cwd=REPO_DIR, check=True)
CODE_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
if CODE_COMMIT != EXPECTED_CODE_COMMIT:
    raise RuntimeError(f"Checkout mismatch: {CODE_COMMIT} != {EXPECTED_CODE_COMMIT}")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", ".[dev,nifti,evaluation]"],
    cwd=REPO_DIR,
    check=True,
)
SOURCE_DIR = str((REPO_DIR / "src").resolve())
sys.path[:] = [entry for entry in sys.path if entry != SOURCE_DIR]
sys.path.insert(0, SOURCE_DIR)
for module_name in tuple(sys.modules):
    if module_name == "fieldbridge" or module_name.startswith("fieldbridge."):
        del sys.modules[module_name]
importlib.invalidate_caches()
import fieldbridge as installed_fieldbridge

PACKAGE_FILE = Path(installed_fieldbridge.__file__).resolve()
if REPO_DIR not in PACKAGE_FILE.parents:
    raise RuntimeError(f"FieldBridge imported from stale location: {PACKAGE_FILE}")
os.chdir(REPO_DIR)


In [ ]:
import json
import shutil

import torch

from fieldbridge.config import load_yaml_config
from fieldbridge.data.volume_splits import (
    audit_volume_splits,
    load_volume_splits,
    volume_splits_fingerprint,
)

CONFIG_PATH = REPO_DIR / "configs/experiment/pseudo_pair_t2flair_duration_probe_10epoch.yaml"
CONFIG = load_yaml_config(CONFIG_PATH)
if CONFIG["training"]["epochs"] != EXPECTED_EPOCHS:
    raise RuntimeError("Duration-probe config no longer declares exactly 10 epochs.")
if CONFIG["evaluation"]["evaluation_after_epoch"] != EXPECTED_EPOCHS:
    raise RuntimeError("Evaluation endpoint must remain epoch 10.")
if CONFIG["training"]["resume_from"] is not None:
    raise RuntimeError("Duration probe must use fresh initialization.")
if not CONFIG["training"]["amp"] or not torch.cuda.is_available():
    raise RuntimeError("This probe requires CUDA with AMP enabled.")
subprocess.run(["nvidia-smi"], check=True)

PRIOR_SPLITS = load_volume_splits(PRIOR_SPLIT_PATH)
audit_volume_splits(PRIOR_SPLITS).raise_for_leakage()
ACTUAL_SPLIT_SHA256 = volume_splits_fingerprint(PRIOR_SPLITS)
if ACTUAL_SPLIT_SHA256 != EXPECTED_SPLIT_SHA256:
    raise RuntimeError(
        f"Split fingerprint mismatch: {ACTUAL_SPLIT_SHA256} != {EXPECTED_SPLIT_SHA256}"
    )

RUN_SPLIT_PATH = RUN_DIR / "volume_splits.json"
shutil.copy2(PRIOR_SPLIT_PATH, RUN_SPLIT_PATH)
if volume_splits_fingerprint(load_volume_splits(RUN_SPLIT_PATH)) != EXPECTED_SPLIT_SHA256:
    raise RuntimeError("Copied split fingerprint changed.")
shutil.copy2(CONFIG_PATH, RUN_DIR / "declared_config.yaml")
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
if CHECKPOINT_DIR.exists():
    raise FileExistsError("Checkpoint directory must not exist before fresh training.")


In [ ]:
def parse_trailing_json(text):
    for start in range(len(text) - 1, -1, -1):
        if text[start] != "{":
            continue
        try:
            return json.loads(text[start:])
        except json.JSONDecodeError:
            pass
    raise ValueError("Command output did not end with a JSON object.")

COMMON_TRAIN_ARGS = [
    sys.executable,
    "-u",
    "-m",
    "fieldbridge.cli",
    "train-pseudo-pairs",
    "--config",
    str(CONFIG_PATH),
    "--manifest",
    str(MANIFEST_PATH),
    "--split-json",
    str(RUN_SPLIT_PATH),
    "--checkpoint-dir",
    str(CHECKPOINT_DIR),
    "--epochs",
    str(EXPECTED_EPOCHS),
    "--workers",
    "0",
]
PREFLIGHT = subprocess.run(
    [*COMMON_TRAIN_ARGS, "--preflight", "--json"],
    cwd=REPO_DIR,
    text=True,
    capture_output=True,
)
(RUN_DIR / "preflight_private.log").write_text(
    PREFLIGHT.stdout + PREFLIGHT.stderr, encoding="utf-8"
)
print(PREFLIGHT.stdout)
if PREFLIGHT.returncode != 0:
    raise RuntimeError(PREFLIGHT.stderr)
PREFLIGHT_JSON = parse_trailing_json(PREFLIGHT.stdout)
if PREFLIGHT_JSON["split_sha256"] != EXPECTED_SPLIT_SHA256:
    raise RuntimeError("Preflight loaded a different split.")
if PREFLIGHT_JSON["steps_per_epoch"] != EXPECTED_STEPS_PER_EPOCH:
    raise RuntimeError("Preflight steps_per_epoch changed from 16.")
if not PREFLIGHT_JSON["leakage_audit"]["ok"]:
    raise RuntimeError("Preflight leakage audit failed.")
if volume_splits_fingerprint(load_volume_splits(RUN_SPLIT_PATH)) != EXPECTED_SPLIT_SHA256:
    raise RuntimeError("Preflight changed the split identity.")


In [ ]:
import csv
import statistics
import time

def nvidia_smi_values(rows, prefix):
    if not rows:
        raise ValueError("nvidia-smi telemetry contains no samples.")
    key = next((name for name in rows[0] if name.startswith(prefix)), None)
    if key is None:
        raise ValueError(f"nvidia-smi telemetry is missing {prefix!r}.")
    values = []
    for row in rows:
        match = re.search(r"-?\d+(?:\.\d+)?", row.get(key, ""))
        if match is not None:
            values.append(float(match.group(0)))
    if not values:
        raise ValueError(f"nvidia-smi telemetry has no numeric {prefix!r} samples.")
    return values

def percentile(values, fraction):
    ordered = sorted(values)
    position = (len(ordered) - 1) * fraction
    lower = int(position)
    upper = min(lower + 1, len(ordered) - 1)
    weight = position - lower
    return ordered[lower] * (1.0 - weight) + ordered[upper] * weight

def summarize_nvidia_smi(rows):
    utilization = nvidia_smi_values(rows, "utilization.gpu")
    memory_used = nvidia_smi_values(rows, "memory.used")
    power_draw = nvidia_smi_values(rows, "power.draw")
    return {
        "gpu_utilization_percent": {
            "mean": statistics.fmean(utilization),
            "median": statistics.median(utilization),
            "p95": percentile(utilization, 0.95),
            "max": max(utilization),
        },
        "memory_used_mib": {"max": max(memory_used)},
        "power_draw_watts": {
            "mean": statistics.fmean(power_draw),
            "max": max(power_draw),
        },
    }

NVIDIA_SMI_FIELDS = (
    "timestamp,utilization.gpu,memory.used,memory.total,power.draw,power.limit"
)
GPU_TELEMETRY_PATH = RUN_DIR / "nvidia_smi_training.csv"
TRAIN_LOG_PATH = RUN_DIR / "train_private.log"
TRAIN_COMMAND = [*COMMON_TRAIN_ARGS, "--json"]

with GPU_TELEMETRY_PATH.open("w", encoding="utf-8") as telemetry_file:
    telemetry = subprocess.Popen(
        [
            "nvidia-smi",
            f"--query-gpu={NVIDIA_SMI_FIELDS}",
            "--format=csv",
            "--loop=5",
        ],
        stdout=telemetry_file,
        stderr=subprocess.STDOUT,
        text=True,
    )
    started = time.perf_counter()
    try:
        with TRAIN_LOG_PATH.open("w", encoding="utf-8") as train_log:
            training = subprocess.Popen(
                TRAIN_COMMAND,
                cwd=REPO_DIR,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
                bufsize=1,
            )
            assert training.stdout is not None
            for line in training.stdout:
                print(line, end="")
                train_log.write(line)
            TRAIN_RETURN_CODE = training.wait()
    finally:
        TRAIN_WALL_SECONDS = time.perf_counter() - started
        telemetry.terminate()
        try:
            telemetry.wait(timeout=15)
        except subprocess.TimeoutExpired:
            telemetry.kill()
            telemetry.wait()

if TRAIN_RETURN_CODE != 0:
    raise RuntimeError(f"Training failed with return code {TRAIN_RETURN_CODE}.")
with GPU_TELEMETRY_PATH.open("r", encoding="utf-8", newline="") as telemetry_input:
    TELEMETRY_ROWS = list(csv.DictReader(telemetry_input))
GPU_TELEMETRY_SAMPLES = len(TELEMETRY_ROWS)
if GPU_TELEMETRY_SAMPLES < 1:
    raise RuntimeError("nvidia-smi did not record utilization, memory, and power samples.")
GPU_TELEMETRY_SUMMARY = summarize_nvidia_smi(TELEMETRY_ROWS)


In [ ]:
from fieldbridge.training.checkpoints import load_checkpoint

LAST_CHECKPOINT = CHECKPOINT_DIR / "last.pt"
if not LAST_CHECKPOINT.is_file():
    raise FileNotFoundError("Training did not produce the epoch-10 last checkpoint.")
STATE = load_checkpoint(LAST_CHECKPOINT, map_location="cpu")
if STATE.get("trainer") != "pseudo_pair_epochs":
    raise RuntimeError("Unexpected trainer identity.")
if int(STATE.get("pseudo_pair_pipeline_version", 0)) != 2:
    raise RuntimeError("Duration probe requires pseudo-pair pipeline v2.")
if int(STATE.get("epoch", -1)) != EXPECTED_EPOCHS:
    raise RuntimeError("Training did not end at epoch 10.")
if int(STATE.get("global_step", -1)) != EXPECTED_GLOBAL_STEPS:
    raise RuntimeError("Fresh duration probe must end at global step 160.")
if STATE.get("run_metadata", {}).get("split_sha256") != EXPECTED_SPLIT_SHA256:
    raise RuntimeError("Checkpoint metadata records a different split identity.")
STEPS_PER_SECOND = EXPECTED_GLOBAL_STEPS / TRAIN_WALL_SECONDS


In [ ]:
# The held-out test evaluation occurs exactly once, at the declared epoch-10 endpoint.
EVAL_COMMAND = [
    sys.executable,
    "-m",
    "fieldbridge.cli",
    "eval-pseudo-pairs",
    "--config",
    str(CONFIG_PATH),
    "--manifest",
    str(MANIFEST_PATH),
    "--checkpoint",
    str(LAST_CHECKPOINT),
    "--split-json",
    str(RUN_SPLIT_PATH),
    "--split",
    "test",
    "--workers",
    "0",
    "--json",
]
EVALUATION_PROCESS = subprocess.run(
    EVAL_COMMAND, cwd=REPO_DIR, text=True, capture_output=True
)
if EVALUATION_PROCESS.returncode != 0:
    raise RuntimeError(EVALUATION_PROCESS.stderr)
EVALUATION = json.loads(EVALUATION_PROCESS.stdout)
(RUN_DIR / "evaluation_private.json").write_text(
    json.dumps(EVALUATION, indent=2, sort_keys=True), encoding="utf-8"
)
if EVALUATION.get("complete_volume") is not False:
    raise RuntimeError("Selected-slice evaluation must report complete_volume false.")
if EVALUATION.get("evidence_scope") != "sampled_slice_per_volume_exploratory":
    raise RuntimeError("Unexpected evaluation evidence scope.")


In [ ]:
VOLUME_EVAL = EVALUATION["sampled_slice_per_volume"]
MACRO = VOLUME_EVAL["macro_average"]
PER_FIELD = VOLUME_EVAL["per_target_field"]
CONDITIONING = VOLUME_EVAL["target_conditioning_audit"]
VOLUME_CONDITIONING = CONDITIONING["volume_level"]
THRESHOLDS = CONFIG["evaluation"]

DEGRADED_NRMSE = float(MACRO["degraded"]["nrmse"])
PREDICTED_NRMSE = float(MACRO["predicted"]["nrmse"])
DEGRADED_SSIM = float(MACRO["degraded"]["ssim"])
PREDICTED_SSIM = float(MACRO["predicted"]["ssim"])
RELATIVE_NRMSE_IMPROVEMENT = (DEGRADED_NRMSE - PREDICTED_NRMSE) / abs(DEGRADED_NRMSE)
ABSOLUTE_SSIM_IMPROVEMENT = PREDICTED_SSIM - DEGRADED_SSIM
FIELDS_WITH_NRMSE_IMPROVEMENT = sum(
    float(field_metrics["predicted"]["nrmse"])
    < float(field_metrics["degraded"]["nrmse"])
    for field_metrics in PER_FIELD.values()
)
FRACTION_CORRECT_BEST = float(VOLUME_CONDITIONING["fraction_volumes_correct_best_nrmse"])
MEAN_MARGIN = float(VOLUME_CONDITIONING["mean_margin_vs_best_wrong_nrmse"])
CORRECT_VS_WRONG_RELATIVE = float(
    CONDITIONING["correct_vs_wrong_improvement"]["relative"]["nrmse"]
)
CORRECT_VS_PERMUTED_RELATIVE = float(
    CONDITIONING["correct_vs_permuted_improvement"]["relative"]["nrmse"]
)
PREDICTED_OUTSIDE_MASK = float(MACRO["predicted"]["outside_mask_mean_abs"])

GATE_DETAILS = {
    "macro_relative_nrmse_improvement": RELATIVE_NRMSE_IMPROVEMENT
    >= float(THRESHOLDS["min_macro_relative_nrmse_improvement"]),
    "macro_absolute_ssim_improvement": ABSOLUTE_SSIM_IMPROVEMENT
    >= float(THRESHOLDS["min_macro_absolute_ssim_improvement"]),
    "fields_with_nrmse_improvement": FIELDS_WITH_NRMSE_IMPROVEMENT
    >= int(THRESHOLDS["min_fields_with_nrmse_improvement"]),
    "fraction_volumes_correct_best_nrmse": FRACTION_CORRECT_BEST
    >= float(THRESHOLDS["min_fraction_volumes_correct_best_nrmse"]),
    "mean_margin_vs_best_wrong_nrmse": MEAN_MARGIN
    >= float(THRESHOLDS["min_mean_margin_vs_best_wrong_nrmse"]),
    "relative_correct_vs_wrong_nrmse": CORRECT_VS_WRONG_RELATIVE
    >= float(THRESHOLDS["min_relative_correct_vs_wrong_nrmse_improvement"]),
    "relative_correct_vs_permuted_nrmse": CORRECT_VS_PERMUTED_RELATIVE
    >= float(THRESHOLDS["min_relative_correct_vs_permuted_nrmse_improvement"]),
    "macro_outside_mask_mean_abs": PREDICTED_OUTSIDE_MASK
    <= float(THRESHOLDS["max_macro_outside_mask_mean_abs"]),
}

HANDOFF = {
    "evidence_source": "user_executed_private_colab",
    "evidence_scope": "sampled_slice_per_volume_exploratory",
    "complete_volume": False,
    "code_commit": CODE_COMMIT,
    "config_name": CONFIG_PATH.name,
    "split_sha256": ACTUAL_SPLIT_SHA256,
    "split_evidence_role": "development_reuse_not_confirmatory",
    "seed": int(CONFIG["seed"]),
    "pipeline_version": int(STATE["pseudo_pair_pipeline_version"]),
    "endpoint": {
        "epoch": int(STATE["epoch"]),
        "global_step": int(STATE["global_step"]),
    },
    "runtime": {
        "cuda": True,
        "amp": True,
        "wall_seconds": TRAIN_WALL_SECONDS,
        "steps_per_second": STEPS_PER_SECOND,
        "gpu_telemetry_samples": GPU_TELEMETRY_SAMPLES,
        "gpu_telemetry_fields": NVIDIA_SMI_FIELDS.split(","),
        "gpu_telemetry": GPU_TELEMETRY_SUMMARY,
    },
    "engineering_gate": "PASS",
    "scientific_gate": "PASS" if all(GATE_DETAILS.values()) else "FAIL",
    "gate_details": GATE_DETAILS,
    "counts": {
        "subjects": int(VOLUME_EVAL["counts"]["subjects"]),
        "volumes": int(VOLUME_EVAL["counts"]["volumes"]),
        "selected_slices": int(VOLUME_EVAL["counts"]["selected_slices"]),
    },
    "per_target_field": {
        field: {
            "volumes": int(field_metrics["volumes"]),
            "selected_slices": int(field_metrics["selected_slices"]),
            "degraded": dict(field_metrics["degraded"]),
            "predicted": dict(field_metrics["predicted"]),
        }
        for field, field_metrics in sorted(PER_FIELD.items())
    },
    "metrics": {
        "degraded_macro_nrmse": DEGRADED_NRMSE,
        "predicted_macro_nrmse": PREDICTED_NRMSE,
        "degraded_macro_ssim": DEGRADED_SSIM,
        "predicted_macro_ssim": PREDICTED_SSIM,
        "fields_with_nrmse_improvement": FIELDS_WITH_NRMSE_IMPROVEMENT,
        "fraction_volumes_correct_best_nrmse": FRACTION_CORRECT_BEST,
        "mean_margin_vs_best_wrong_nrmse": MEAN_MARGIN,
        "relative_correct_vs_wrong_nrmse": CORRECT_VS_WRONG_RELATIVE,
        "relative_correct_vs_permuted_nrmse": CORRECT_VS_PERMUTED_RELATIVE,
        "macro_outside_mask_mean_abs": PREDICTED_OUTSIDE_MASK,
    },
    "scaled_pilot": "BLOCKED_PENDING_REVIEW",
    "limitations": [
        "selected slices only",
        "observed development split",
        "not confirmatory evidence",
        "not complete-volume evidence",
        "not learned real 0.1T translation evidence",
    ],
}

FORBIDDEN_HANDOFF_TERMS = (
    "subject_id",
    "volume_path",
    "record_id",
    "slice_index",
    ".nii",
    "/content/drive",
    "last.pt",
    "best.pt",
    "image",
)
HANDOFF_TEXT = json.dumps(HANDOFF, indent=2, sort_keys=True)
if any(term in HANDOFF_TEXT.lower() for term in FORBIDDEN_HANDOFF_TERMS):
    raise RuntimeError("Sanitized handoff contains a forbidden identity or artifact term.")
(RUN_DIR / "codex_handoff_sanitized.json").write_text(HANDOFF_TEXT, encoding="utf-8")
print(HANDOFF_TEXT)


## Stop here

Do not launch the scaled pilot from this notebook. Review the sanitized duration-probe handoff against the frozen viability gates. Even a passing result remains development evidence because the split was previously observed and `complete_volume` is false.